In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

# LoRA Finetune

## Merge

Merge the checkpoint with the base model and create a full model with the fine tune.

In [3]:
base_model = "HuggingFaceTB/SmolLM-135M"
checkpoint_path = "../../runs/3/checkpoint-116400"
model_path = "./lora-full-model"

In [4]:
# ---- 1. Load base model ----
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype="auto",
    device_map="auto"
)

# ---- 2. Load LoRA adapter ----
model = PeftModel.from_pretrained(model, checkpoint_path)

# ---- 3. Merge LoRA into base weights ----
model = model.merge_and_unload()

# ---- 4. Save merged MODEL only ----
model.save_pretrained(
    model_path,
    safe_serialization=True
)

# ---- 5. CRITICAL: reload tokenizer from BASE (clean source of truth) ----
tokenizer = AutoTokenizer.from_pretrained(base_model)

# ensure padding token exists (important for GPT-style models)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---- 6. Save tokenizer cleanly ----
tokenizer.save_pretrained(model_path)

print("Model saved to:", model_path)

Model saved to: ./lora-full-model


## Onnxize

Convert the full model to an ONNX model.
This will allow us to run the model in the onnx web runtime which transformer.js uses.

In [9]:
from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

ort_model = ORTModelForCausalLM.from_pretrained(model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path)

ort_model.save_pretrained(os.path.join(model_path, "onnx"))
tokenizer.save_pretrained(os.path.join(model_path, "onnx"))

No ONNX files were found for ./lora-full-model, setting `export=True` to convert the model to ONNX. Don't forget to save the resulting model with `.save_pretrained()`
Found different candidate ONNX initializers (likely duplicate) for the tied weights:
	lm_head.weight: {'onnx::MatMul_9817'}
	model.embed_tokens.weight: {'model.embed_tokens.weight'}


('./lora-full-model/onnx/tokenizer_config.json',
 './lora-full-model/onnx/special_tokens_map.json',
 './lora-full-model/onnx/vocab.json',
 './lora-full-model/onnx/merges.txt',
 './lora-full-model/onnx/added_tokens.json',
 './lora-full-model/onnx/tokenizer.json')

## Upload to Huggingface

In [ ]:
# To make it readable by transformers.js, we need to add the model under ./onnx/model.safetensors
# Otherwise it won't find and load the model.
import shutil

src = os.path.join("onnx", "model.onnx")

os.makedirs(os.path.join(model_path, "onnx"), exist_ok=True)
dst = os.path.join(model_path, "onnx", "model.onnx")

shutil.copyfile(src, dst)

os.makedirs(os.path.join("onnx", "onnx"), exist_ok=True)
shutil.copyfile(src, dst)

'./lora-full-model/onnx/model.onnx'

In [ ]:
from huggingface_hub import upload_folder, login
login("redacted")

upload_folder(
    repo_id="throvn/abstract-title-generator",
    folder_path="lora-full-model",
    commit_message="force update",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Throvn/abstract-title-generator/commit/d3b52a183659be2ad39c0dc8671113828fbff936', commit_message='force update', commit_description='', oid='d3b52a183659be2ad39c0dc8671113828fbff936', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Throvn/abstract-title-generator', endpoint='https://huggingface.co', repo_type='model', repo_id='Throvn/abstract-title-generator'), pr_revision=None, pr_num=None)